In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00-common/01.environment-config"

In [0]:
%run "../00-common/02.bronze-helpers"

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/results/"

target_path = "s3://bizf1project123/Bronze/Aaisha/incremental-load/results"

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    FloatType,
    DateType
)

results_schema = StructType([
    StructField("date", DateType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])


In [0]:
results_df = (
    spark.read
        .format("json")
        .schema(results_schema)
        .option("mode", "FAILFAST")
        .load(source_file)
)


In [0]:
display(results_df)

In [0]:
results_final_df = add_ingestion_metadata(results_df)

In [0]:
write_to_bronze(
    results_final_df,
    target_path,
    v_batch_id
)